# Cadastro Nacional de Unidades de Conservação

O governo brasileiro protege as áreas naturais por meio de Unidades de Conservação (UC) - estratégia extremamente eficaz para a manutenção dos recursos naturais em longo prazo. Para atingir esse objetivo de forma efetiva e eficiente, foi instituído o Sistema Nacional de Conservação da Natureza (SNUC), com a promulgação da [Lei nº 9.985, de 18 de julho de 2000](https://www.planalto.gov.br/ccivil_03/leis/l9985.htm). A Lei do SNUC representou grandes avanços à criação e gestão das UC nas três esferas de governo (federal, estadual e municipal), pois ele possibilita uma visão de conjunto das áreas naturais a serem preservadas. Além disso, estabeleceu mecanismos que regulamentam a participação da sociedade na gestão das UC, potencializando a relação entre o Estado, os cidadãos e o meio ambiente.

Um dos instrumentos da Lei do SNUC é o [Cadastro Nacional de Unidades de Conservação (CNUC)](https://cnuc.mma.gov.br) e sua gestão ocorre pelo [Departamento de Áreas Protegidas (DAP)](https://www.gov.br/mma/pt-br/composicao/sbio/dap) do _Ministério do Meio Ambiente_ (MMA).

<br>

A vantagem de obter os dados do CNUC é que, supostamente, todas as UCs Estaduais e Municipais também estão cadastradas. Quando obtemos os dados do ICMBio, apenas as UCs Federais estarão listadas.


In [ ]:
import io
import tempfile
from pathlib import Path

import geopandas as gpd
from owslib.wfs import WebFeatureService
from owslib.wms import WebMapService

import open_geodata as geo

<br>

---

## Pooch

Os dados do CNUC está disponíveis de formas diferentes: a primeira delas é o [Portal de Dados Abertos do Governo Federal](https://dados.gov.br/dados/conjuntos-dados/unidadesdeconservacao). E, portanto, com auxílio do [pooch](https://www.fatiando.org/pooch/latest/) foi possível baixar.


In [ ]:
db = geo.data.DB(db='br_cnuc')
db.list_data

In [ ]:
filename = db.get_data(name='geo.Polígono CNUC 2025_03')
filename

In [ ]:
gdf = geo.load_dataset(
    db='br_cnuc',
    name='geo.Polígono CNUC 2025_03',
    shapefile='cnuc_2025_03.shp',
    engine='fiona',
)
gdf.info()
gdf.head()

In [ ]:
gdf.explore(column='categoria')

<br>

Consumindo os dados tabulares

In [ ]:
df = geo.load_dataset(db='br_cnuc', name='tab.CNUC_2025_1º semestre', sep=';')
df.info()
df.head()

<br>

-----

## Outros

DESENVOLVER. NÃO TIVE SUCESSO ATÉ O MOMENTO.

A outra forma é explorar o MapServer que dá sustentação ao portal do CNUC.

> https://demo.mapserver.org/cgi-bin/msautotest?SERVICE=WMS&VERSION=1.3.0&REQUEST=GetCapabilities

> https://cnuc-mapserv.mma.gov.br/cgi-bin/mapserv?MAP=/var/www/storage/app/mapfiles/ucs.map&SERVICE=WMS&VERSION=1.3.0&REQUEST=GetMap&FORMAT=image/png&TRANSPARENT=true&LAYERS=ucs&ucIds=undefined&WIDTH=256&HEIGHT=256&CRS=EPSG:3857&STYLES=&BBOX=-5009377.085697312,-7514065.628545966,-2504688.5428486564,-5009377.08569731


> https://cnuc-mapserv.mma.gov.br/cgi-bin/mapserv?SERVICE=WMS&VERSION=1.3.0&REQUEST=GetCapabilities

In [ ]:
wfs = WebFeatureService(
    # url='https://geoserver.funai.gov.br/geoserver/ows/',
    # url='http://mapas.mma.gov.br/cgi-bin/mapserv?map=/opt/www/html/webservices/biorregioes.map&SERVICE=WMS&REQUEST=GetCapabilities',
    url='http://mapas.mma.gov.br/cgi-bin/mapserv?map=/opt/www/html/webservices/florestaspublicas.map&',
    #
    # version='1.3.0',
    #version='2.0.0',
)

In [ ]:
wms = WebMapService(
    # url='https://geoserver.funai.gov.br/geoserver/ows/',
    # url='http://mapas.mma.gov.br/cgi-bin/mapserv?map=/opt/www/html/webservices/biorregioes.map',
    # url='http://mapas.mma.gov.br/cgi-bin/mapserv?map=/opt/www/html/webservices/florestaspublicas.map&',
    url='http://mapas.mma.gov.br/i3geo/ogc.php?tema=undefined',  #  i3geo
    #
    version='1.3.0',
    # version='2.0.0',
)

In [ ]:
for layer_name, layer in wms.contents.items():
    # if layer.queryable == 0:
    print(f"Layer: {layer_name}")
    print(f"  Title: {layer.title}")
    print(f"  Abstract: {layer.abstract}")
    print(f"  BoundingBox: {layer.boundingBoxWGS84}")
    print(f"  CRS: {layer.crsOptions}")
    print(f"  Styles: {layer.styles}")
    print(f"  Keywords: {layer.keywords}")
    # print(f"  Queryable: {layer.queryable}")
    # print(f"  Opaque: {layer.opaque}")
    # print(f"  Dimensions: {layer.dimensions}")
    print(f"  MetadataURLs: {layer.metadataUrls}")
    print()

In [ ]:
# Obter os dados no formato GeoJSON (ou outro formato suportado)
response = wms.getfeature(
    typename='estadosl',
    # bbox=(173700, 440400, 178700, 441400),
    # srsname='EPSG:28992'
    # srsname='EPSG:4326',
    # srsname='EPSG:4674',
    # outputFormat='application/json',
)
response

<br>

https://www.gov.br/icmbio/pt-br/assuntos/dados_geoespaciais/mapa-tematico-e-dados-geoestatisticos-das-unidades-de-conservacao-federais